# P5-1. 미니프로젝트 템플릿 — 베이스라인 확보

**PART 5 · DNN 미니프로젝트 · 학생용 (Student)**

---

## 학습 목표

1. 프로젝트 표준 구조를 그대로 가져다 쓴다
2. 0·1·2단계 베이스라인을 순서대로 확보한다
3. 평가 지표와 성공 기준을 코드로 고정한다


## 이 노트북 사용법 — 학생용 실습본

이 파일은 **핵심 코드만** 빈칸(`____`)으로 비워 둔 실습본입니다.

1. 코드 안의 `①`, `②` 번호와 힌트를 먼저 읽습니다.
2. `____`를 알맞은 값이나 코드로 바꿉니다.
3. `Shift + Enter`로 셀을 실행합니다.

| 오류 메시지 | 원인 |
|---|---|
| `NameError: name '____' is not defined` | 아직 채우지 않은 빈칸이 남아 있습니다. |
| `SyntaxError` | 빈칸과 함께 괄호나 따옴표를 지웠습니다. |

> `____`가 아닌 부분은 실행을 돕는 보조 코드입니다. 처음에는 수정하지 않아도 됩니다.


## 사용 방법

이 노트북은 **틀**이다. 아래 `CONFIG` 만 팀의 과제에 맞게 바꾸면 나머지는 그대로 돌아간다.
데이터 로드 부분만 교체하고 이후 절차는 손대지 않는 것이 원칙이다.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import tensorflow as tf
layers = tf.keras.layers

plt.rcParams["axes.unicode_minus"] = False
tf.keras.utils.set_random_seed(42)
np.random.seed(42)
RANDOM_STATE = 42


> **입문자 안내**
> 내부 구현 전체를 외우지 않습니다. `____`가 있는 핵심 설정과 실행 순서에 집중하고, 긴 함수·클래스 코드는 실행용 보조 코드로 사용하세요.


## 0. 프로젝트 정의 — 코드를 쓰기 전에 문장으로 적는다

In [5]:
CONFIG = {
# ① 아래 힌트를 보고 핵심 코드를 완성하세요.
    "project_name": "customer",        # 팀 과제명을 적는다
    "task_type": "binary",                   # binary / multiclass / regression
    "target": "churn",                       # 목표 변수명
# ② 아래 힌트를 보고 핵심 코드를 완성하세요.
    "primary_metric": "accuracy",                  # 주 평가 지표 (accuracy/f1/recall/mae 등)
# ③ 아래 힌트를 보고 핵심 코드를 완성하세요.
    "success_criterion": "accuracy가 가장 높은 모델을 선정하고 고객이탈여부를 예측한다.",   # 성공 기준을 문장으로 적는다
    "team": ["팀원A", "팀원B", "팀원C"],
    "test_size": 0.15,
    "val_size": 0.18,
}

print("=" * 60)
for k, v in CONFIG.items():
    print(f"{k:20s} : {v}")
print("=" * 60)


project_name         : customer
task_type            : binary
target               : churn
primary_metric       : accuracy
success_criterion    : accuracy가 가장 높은 모델을 선정하고 고객이탈여부를 예측한다.
team                 : ['팀원A', '팀원B', '팀원C']
test_size            : 0.15
val_size             : 0.18


> **작성 규칙** — 아래 네 문장을 팀이 직접 채운다. 코드보다 먼저 한다.
>
> 1. 우리는 **( )** 를 입력으로 **( )** 를 예측한다.
> 2. 성공은 **( )** 지표가 **( )** 이상일 때로 정의한다.
> 3. 이 문제에서 더 비싼 실수는 **(FN / FP)** 이다. 그 이유는 **( )** 이다.
> 4. 우리가 넘어야 할 기준선은 **( )** 이다.

## 1. 데이터 로드 — 이 셀만 팀 과제에 맞게 교체한다

In [ ]:
# df = pd.read_csv("data_v2_save.csv")

###원본 CSV
   │
   ├─ Churn → y (예측 정답)
   │
   └─ 나머지 컬럼 → X
                 │
                 ↓
          get_dummies()
                 │
                 ├─ gender → 0/1
                 ├─ Partner → 0/1
                 ├─ InternetService → 여러 개의 0/1 컬럼
                 ├─ Contract → 여러 개의 0/1 컬럼
                 ├─ PaymentMethod → 여러 개의 0/1 컬럼
                 └─ 기존 숫자형 변수 → 그대로 유지
                 │
                 ↓
              float32
                 │
                 ↓
          신경망 모델 입력

In [6]:
# ------------------------------------------------------------------
# [교체 구간] 팀 데이터로 바꾼다.
#   df = pd.read_csv("우리팀_데이터.csv")
#   X = df.drop(columns=[CONFIG["target"]]).values.astype("float32")
#   y = df[CONFIG["target"]].values
# ------------------------------------------------------------------
# 1. CSV 파일 불러오기
df = pd.read_csv("data_v2_save.csv")

# 2. 예측할 목표 변수 설정
CONFIG["target"] = "Churn"

# 3. X(입력 데이터)와 y(정답 데이터) 분리
X_df = df.drop(columns=[CONFIG["target"]])
y = df[CONFIG["target"]].values

# 4. 문자형 범주형 데이터를 숫자로 변환
X_df = pd.get_dummies(X_df, drop_first=True)

# 5. X를 float32 형태의 숫자 배열로 변환
X = X_df.values.astype("float32")
y = y.astype("float32")

# 특성 이름 저장
feature_names = X_df.columns.tolist()

# ------------------------------------------------------------------
# 데이터 확인
# ------------------------------------------------------------------

print("데이터 형태:", df.shape)
print("전처리 후 X 형태:", X.shape)

print("결측치:", df.isnull().sum().sum(), "건")

print("\n목표 변수 분포:")
u, cnt = np.unique(y, return_counts=True)

for k, v in zip(u, cnt):
    print(f"  {int(k)}: {v:5d}건 ({v/len(y):.1%})")

df.head()


데이터 형태: (7027, 17)
전처리 후 X 형태: (7027, 26)
결측치: 0 건

목표 변수 분포:
  0:  5161건 (73.4%)
  1:  1866건 (26.6%)


,gender,Partner,Dependents,tenure,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,No,No,34,No,DSL,Yes,No,No,No,No,One year,No,Mailed check,56.95,1889.50,0
1,Male,No,No,2,No,DSL,Yes,Yes,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
2,Male,No,No,45,No phone service,DSL,Yes,No,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
3,Female,No,No,2,No,Fiber optic,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
4,Female,No,No,8,Yes,Fiber optic,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1


## 2. 분할 — 한 번만 나누고 끝까지 유지한다

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 1. Train / Test 분할
# ------------------------------------------------------------

# 분류 문제이므로 y의 비율을 유지하면서 나눈다.
stratify = y if CONFIG["task_type"] != "regression" else None

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=CONFIG["test_size"],
    random_state=RANDOM_STATE, stratify=stratify)
    
# ------------------------------------------------------------
# 2. Train / Validation 분할
# ------------------------------------------------------------
stratify_temp = y_temp if CONFIG["task_type"] != "regression" else None
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=CONFIG["val_size"],
    random_state=RANDOM_STATE, stratify=stratify_temp)

# ------------------------------------------------------------
# 3. 표준화(Standard Scaling)
# ------------------------------------------------------------
scaler = StandardScaler()

# ⚠️scaler는 X_train에만 fit
X_train = scaler.fit_transform(X_train).astype("float32")   # 학습으로만 fit
X_val = scaler.transform(X_val).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

print(f"train {X_train.shape}  ({len(X_train)/len(X):.0%})")
print(f"val   {X_val.shape}  ({len(X_val)/len(X):.0%})")
print(f"test  {X_test.shape}  ({len(X_test)/len(X):.0%})")
print("\n주의: test는 마지막 한 번만 사용한다. 튜닝에는 val만 쓴다.")


## 3. 평가 함수 — 지표를 코드로 고정한다

지표 계산을 함수 하나로 고정해야 모든 실험이 같은 기준으로 비교된다.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, mean_absolute_error,
                             mean_squared_error, r2_score)


def evaluate(y_true, y_prob, threshold=0.5, task=None):
    """과제 유형에 맞는 지표를 dict로 반환한다."""
    task = task or CONFIG["task_type"]

    if task == "regression":
        pred = np.asarray(y_prob).ravel()
        return {
            "mae": round(mean_absolute_error(y_true, pred), 4),
            "rmse": round(float(np.sqrt(mean_squared_error(y_true, pred))), 4),
            "r2": round(r2_score(y_true, pred), 4),
        }

    prob = np.asarray(y_prob).ravel()
    pred = (prob > threshold).astype(int)
    return {
        "accuracy": round(accuracy_score(y_true, pred), 4),
        # 힌트: 위 accuracy와 같은 형태로 채운다 (zero_division=0 유지)
        "precision": ____,   # ④
        "recall": ____,  # ⑤
        "f1": ____,  # ⑥
        "roc_auc": round(roc_auc_score(y_true, prob), 4),
        "pr_auc": round(average_precision_score(y_true, prob), 4),
    }


def show_confusion(y_true, y_prob, threshold=0.5, title=""):
    pred = (np.asarray(y_prob).ravel() > threshold).astype(int)
    cm = confusion_matrix(y_true, pred)
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["예측 0", "예측 1"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["실제 0", "실제 1"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=15,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_title(f"혼동행렬 {title}")
    plt.colorbar(im, shrink=0.8); plt.tight_layout(); plt.show()
    return cm


print("평가 함수 준비 완료. 주 지표:", CONFIG["primary_metric"])


## 4. 0단계 베이스라인 — 무지성 기준

**아무것도 배우지 않은 모델**의 성능이다. 이보다 못하면 모델이 실패한 것이다.

In [ ]:
from sklearn.dummy import DummyClassifier, DummyRegressor

if CONFIG["task_type"] == "regression":
    dummy = DummyRegressor(strategy="mean")
else:
    dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(X_train, y_train)

if CONFIG["task_type"] == "regression":
    prob0 = dummy.predict(X_val)
else:
    prob0 = dummy.predict_proba(X_val)[:, 1]

score_0 = evaluate(y_val, prob0)
print("0단계 · 무지성 기준")
for k, v in score_0.items():
    print(f"  {k:10s} {v}")
print("\n→ 정확도가 높아 보여도 재현율이 0이면 아무 의미가 없다")


## 5. 1단계 베이스라인 — 비딥러닝 기준

**딥러닝이 이겨야 할 상대**이다. 정형 데이터에서는 이 기준을 넘기가 의외로 어렵다.

In [ ]:
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
import time

baselines = {}
if CONFIG["task_type"] == "regression":
    candidates = {"Ridge": Ridge()}
else:
    candidates = {
        "로지스틱 회귀": LogisticRegression(max_iter=2000,
                                       class_weight="balanced"),
        "랜덤 포레스트": RandomForestClassifier(n_estimators=300,
                                          class_weight="balanced",
                                          random_state=RANDOM_STATE, n_jobs=-1),
        "히스토그램 GBM": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    }

rows = []
for name, clf in candidates.items():
    t0 = time.time()
    clf.fit(X_train, y_train)
    sec = time.time() - t0
    prob = (clf.predict(X_val) if CONFIG["task_type"] == "regression"
            else clf.predict_proba(X_val)[:, 1])
    sc = evaluate(y_val, prob)
    sc.update({"모델": name, "학습시간": round(sec, 2)})
    rows.append(sc)
    baselines[name] = (clf, prob)
    print(f"{name:14s} {CONFIG['primary_metric']}={sc[CONFIG['primary_metric']]}")

df_base = pd.DataFrame(rows).set_index("모델")
df_base


In [ ]:
best_ml_name = df_base[CONFIG["primary_metric"]].idxmax()
score_1 = df_base.loc[best_ml_name].to_dict()
print(f"1단계 베이스라인: {best_ml_name}")
print(f"  {CONFIG['primary_metric']} = {score_1[CONFIG['primary_metric']]}")
print(f"\n신경망은 이 값을 넘어야 의미가 있다.")


## 6. 2단계 베이스라인 — 최소 신경망

**정규화 없이, 기본 설정 그대로.** 이 숫자가 이후 모든 개선 실험의 비교 대상이 된다.

In [ ]:
def build_model(hidden=(64, 32), dropout=0.0, use_bn=False, l2=0.0,
                lr=0.001, n_features=None, task=None):
    """프로젝트 표준 모델 빌더. 실험에서 인자만 바꿔 호출한다."""
    n_features = n_features or X_train.shape[1]
    task = task or CONFIG["task_type"]
    reg = tf.keras.regularizers.l2(l2) if l2 > 0 else None

    lyrs = [layers.Input(shape=(n_features,))]
    for h in hidden:
        lyrs.append(layers.Dense(h, kernel_regularizer=reg))
        if use_bn:
            lyrs.append(layers.BatchNormalization())
        lyrs.append(layers.Activation("relu"))
        if dropout > 0:
            lyrs.append(layers.Dropout(dropout))

    if task == "binary":
        lyrs.append(layers.Dense(1, activation="sigmoid"))
        loss, metrics = "binary_crossentropy", ["accuracy"]
    elif task == "multiclass":
        n_cls = int(len(np.unique(y_train)))
        lyrs.append(layers.Dense(n_cls, activation="softmax"))
        loss, metrics = "sparse_categorical_crossentropy", ["accuracy"]
    else:
        lyrs.append(layers.Dense(1))
        loss, metrics = "mse", ["mae"]

    m = tf.keras.Sequential(lyrs)
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss=loss, metrics=metrics)
    return m


def make_callbacks(name="run", patience_es=15, patience_lr=7,
                   monitor="val_loss"):
    return [
        tf.keras.callbacks.EarlyStopping(monitor=monitor, patience=patience_es,
                                      restore_best_weights=True, verbose=0),
        tf.keras.callbacks.ModelCheckpoint(f"{name}_best.keras", monitor=monitor,
                                        save_best_only=True, verbose=0),
        tf.keras.callbacks.ReduceLROnPlateau(monitor=monitor, factor=0.5,
                                          patience=patience_lr, min_lr=1e-6),
        tf.keras.callbacks.CSVLogger(f"{name}_history.csv"),
    ]


print("모델 빌더와 콜백 준비 완료 (P3-2 템플릿 재사용)")


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)
nn_base = build_model()          # 정규화 없음, 기본 설정
nn_base.summary()

hist_base = nn_base.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=100, batch_size=32, verbose=0)

prob_nn = nn_base.predict(X_val, verbose=0).ravel()
score_2 = evaluate(y_val, prob_nn)
print("\n2단계 · 최소 신경망")
for k, v in score_2.items():
    print(f"  {k:10s} {v}")


## 7. 세 단계 베이스라인 정리 — 프로젝트의 출발선

In [ ]:
metric = CONFIG["primary_metric"]
baseline_table = pd.DataFrame([
    {"단계": "0 · 무지성 기준", "모델": "최빈 클래스", metric: score_0[metric]},
    {"단계": "1 · 비딥러닝", "모델": best_ml_name, metric: score_1[metric]},
    {"단계": "2 · 최소 신경망", "모델": "Dense 64-32", metric: score_2[metric]},
])
baseline_table.to_csv("baseline_scores.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(baseline_table["단계"], baseline_table[metric],
              color=["#94A3B8", "#2563EB", "#059669"])
for b, v in zip(bars, baseline_table[metric]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.3f}",
            ha="center", fontsize=12, fontweight="bold")
ax.set_ylabel(metric); ax.set_ylim(0, 1.05)
ax.set_title("베이스라인 3단계 — 이 선을 넘어야 개선이다")
ax.grid(alpha=0.3, axis="y"); plt.tight_layout(); plt.show()

baseline_table


In [ ]:
# 개선의 기준값을 파일로 고정한다 — P5-2에서 읽어 쓴다
import json

target = max(score_1[metric], score_2[metric])
reference = {
    "project": CONFIG["project_name"],
    "metric": metric,
    "success_criterion": CONFIG["success_criterion"],
    "baseline_0_dummy": score_0[metric],
    "baseline_1_ml": score_1[metric],
    "baseline_1_model": best_ml_name,
    "baseline_2_nn": score_2[metric],
    "target_to_beat": round(target, 4),
}
with open("baseline_reference.json", "w", encoding="utf-8") as f:
    json.dump(reference, f, ensure_ascii=False, indent=2)

print(json.dumps(reference, ensure_ascii=False, indent=2))
print(f"\n앞으로 모든 실험은 {metric} = {target:.4f} 를 넘어야 한다.")


### 확인 질문

- 0단계 베이스라인의 재현율은 왜 0인가
- 트리 모델이 신경망보다 좋게 나왔다. 프로젝트를 어떻게 진행하겠는가
- test 세트를 지금 확인하면 안 되는 이유는 무엇인가
- CONFIG의 성공 기준을 팀이 직접 정한 근거는 무엇인가

---

## 정리

- 코드보다 문제 정의가 먼저다. 네 문장을 채우고 시작한다.
- 분할은 한 번만 하고 test는 마지막에 한 번만 쓴다.
- 베이스라인은 3단계(무지성 → 비딥러닝 → 최소 신경망)로 세운다.
- 기준값을 파일로 고정해 두면 개선 여부를 다툴 일이 없다.

**다음 파일** — `P5-2_실험기록표_개선루프.ipynb`

---

## 도전 과제

팀 데이터를 넣어 0·1·2단계 베이스라인을 확보하라. 세 숫자가 순서대로 올라가지 않는다면 그 원인을 찾아 기록한다.